# FSL-SAGE on Colab GPU

Clones this repo from GitHub, installs dependencies, and runs a quick GPU smoke test
(see `docs/part0-mnist-smoke-test.md` for the equivalent CPU run this mirrors).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

Datasets (`datas/`) and run outputs (`saves/`) are stored on your Google Drive so they
persist across Colab session resets instead of re-downloading/re-running every time.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change BRANCH once this work lands on master.
REPO_URL = "https://github.com/juniorfelix998/FSL-SAGE.git"
BRANCH = "ft/add-mnist"

WORKSPACE = "/content/drive/MyDrive/fsl-sage-colab"
REPO_DIR = f"{WORKSPACE}/FSL-SAGE"

import os
os.makedirs(WORKSPACE, exist_ok=True)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

In [ ]:
# Which cut to use for the cells below -- "shallow", "middle" (default,
# this harness's original behavior), or "deep". See CLAUDE.md's
# "Cuts: early/middle/late" benchmark dimension and
# src/hydra_config/cut/*.yaml. Change this one line to re-run every cell
# below at a different cut.
CUT = "middle"

# Override the number of clients (harness default is 10) -- set to an int
# (e.g. 2) or leave as None to use the default. Affects every cell below.
NUM_CLIENTS = None
NUM_CLIENTS_OVERRIDE = f"num_clients={NUM_CLIENTS}" if NUM_CLIENTS is not None else ""
NUM_CLIENTS_LIST_ARG = f"--num_clients_list {NUM_CLIENTS}" if NUM_CLIENTS is not None else ""

In [ ]:
# src/main.py resolves datasets/saves as '../datas' and '../saves' relative to
# src/, so point those at persistent Drive folders instead of Colab's ephemeral disk.
DRIVE_DATAS = f"{WORKSPACE}/datas"
DRIVE_SAVES = f"{WORKSPACE}/saves"
os.makedirs(DRIVE_DATAS, exist_ok=True)
os.makedirs(DRIVE_SAVES, exist_ok=True)

for name, target in (("datas", DRIVE_DATAS), ("saves", DRIVE_SAVES)):
    link = f"{REPO_DIR}/{name}"
    if os.path.islink(link) or os.path.exists(link):
        continue
    os.symlink(target, link)

In [ ]:
# torch/torchvision are pinned to Colab's OWN already-installed versions (not the
# repo's local conda_env.yaml pin of torch==2.5.1) so pip has no reason to touch them --
# swapping torch pulls in a different CUDA toolkit than the one Colab's preinstalled
# RAPIDS stack (cuml/cudf/libraft/libcuvs/cuda-python) was built against, which is what
# caused the wall of "cuda-toolkit ... incompatible" resolver errors. requests is bumped
# to 2.32.4 to match what google-colab/google-adk already require, for the same reason.
# Check !python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__)"
# on a fresh runtime if these ever drift from what Colab ships.
!pip install -q torch==2.13.0 torchvision==0.28.0 hydra-core==1.3.2 hydra-joblib-launcher==1.2.0 \
  omegaconf==2.3.0 wandb==0.19.3 numpy==2.1.3 pandas==2.2.3 scipy==1.14.1 matplotlib==3.9.2 \
  h5py==3.12.1 pyyaml==6.0.2 tqdm==4.67.0 requests==2.32.4 pillow==11.0.0 prettytable==3.12.0 \
  joblib==1.4.2 antlr4-python3-runtime==4.9.3 gitpython==3.1.43

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist cut={CUT} {NUM_CLIENTS_OVERRIDE}

## Running a single method standalone

Supported `algorithm=` keys: `fed_avg`, `sl_multi_server` (SplitFedv1),
`sl_single_server` (SplitFedv2), `vanilla_sl`, `cse_fsl`, `fsl_sage`, `ho_sfl`,
`mu_splitfed`, `dsl_aux`, `han_locloss`, `fedsplitx`, `hosl`, `locfedmix_sl`.

**`mu_splitfed` provenance note:** this is HKU-WILL-Lab/HO-SFL's own third-party
CV/ResNet18 reimplementation of MU-SplitFed, not the original Johnny-Zip/MU-SplitFed
authors' code (their published repo is LLM-only and non-functional as published --
see `src/algos/mu_splitfed.py` for details). Treat any MU-SplitFed numbers accordingly.

Each cell below runs one method for `rounds=3` (a quick sanity check, not a real
result) so you can test any single method on its own without running the full sweep.

**`dsl_aux` provenance note:** AI-assisted no-code reimplementation of DSL-Aux (arXiv:2601.19261), adapted from a partial third-party reference (juniorfelix998/sl-fl-dgl) -- not validated against the paper's own reported numbers. It is also non-federated (single client/server split, no weight aggregation); `num_clients=1` is its paper-faithful setting, though the cell below runs it with the notebook's usual default like every other method for consistency.
**`vanilla_sl` note:** the original, non-federated sequential split-learning
protocol (Gupta & Raskar 2018) -- one shared client model and one shared server
model, no FedAvg step (there's only one copy of each to begin with).

**`han_locloss` provenance note:** AI-assisted no-code reimplementation of Han et
al., "Accelerating FL with SL on Locally Generated Losses" (FL-ICML 2021) -- not
validated against the paper's own reported numbers.

**`fedsplitx` provenance note:** AI-assisted no-code reimplementation of FedSplitX
(arXiv:2310.14579), run at a single shared cut (this benchmark's protocol) rather
than the paper's own multiple simultaneous depth-levels -- not validated against
the paper's own reported numbers.

**`hosl` provenance note:** AI-assisted no-code reimplementation of HOSL
(arXiv:2601.10940) -- distinct from `ho_sfl` above (different paper, confirmed
different arXiv id); not validated against the paper's own reported numbers.

**`locfedmix_sl` provenance note:** AI-assisted no-code reimplementation of
LocFedMix-SL (ACM WWW 2022) -- not validated against the paper's own reported
numbers.

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=fed_avg cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=sl_multi_server cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=sl_single_server cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=cse_fsl cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=fsl_sage cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=ho_sfl cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=mu_splitfed cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=dsl_aux cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=vanilla_sl cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=han_locloss cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=fedsplitx cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=hosl cut={CUT} {NUM_CLIENTS_OVERRIDE}

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=locfedmix_sl cut={CUT} {NUM_CLIENTS_OVERRIDE}

## Running everything: sweep + measurement table + plots in one command

`run_mnist_benchmark.py` is the "one main" entry point: it sweeps all 12 tabled
methods (SplitFedv1, SplitFedv2, Vanilla-SL, CSE-FSL, FSL-SAGE, HO-SFL,
MU-SplitFed, DSL-Aux, Han-et-al, FedSplitX, HOSL, LocFedMix-SL) across both
MNIST distributions (IID and Dirichlet alpha=0.5),
builds the measurement table (comm cut/weights/total, cut vs. weights
share, accuracy, latency, peak memory),
and generates accuracy/communication-load plots -- all from a single command.

Runs at the `CUT` cell's cut by default; pass `--cuts shallow middle deep`
(3x the sweep cost) to compare a method's sensitivity to cut depth --
each cut gets its own table (`benchmark_table_mnist_<cut>.txt`) and plots
directory (`plots_mnist_<cut>/`).

Similarly, pass `--num_clients_list 2 10 100` to sweep multiple client
counts (each gets a `_nc<N>` suffix on its table/plots) -- useful since
some methods (e.g. MU-SplitFed) are known to be sensitive to client count.

**At `--rounds 3` (the default below) this is a pipeline/plumbing check, not a
reportable result** -- it confirms every method runs end-to-end and produces a
`results.json` the table/plot code can parse, not real accuracy numbers. For real
numbers, raise `--rounds` (README/`config.yaml` default is 200) and, per
`CLAUDE.md`'s "Seeds: 3 default", repeat across 3 seeds. At that scale a single
free-tier Colab session likely won't finish in one sitting -- use `--methods` to
re-invoke this for a subset of methods across multiple sessions; each sweep run is
additive (new timestamped folders under `saves/`), nothing gets overwritten.

In [ ]:
%cd {REPO_DIR}/inference
!python run_mnist_benchmark.py --rounds 3 --seed 200 --device cuda --cuts {CUT} {NUM_CLIENTS_LIST_ARG}

In [ ]:
# Inline display of the table + plots, so results are visible without downloading
# anything from Drive. Matches benchmark_table.py's cut/num_clients-aware naming:
# the 'middle' cut + default num_clients keeps the original filenames; shallow/deep
# add a `_<cut>` suffix and an explicit NUM_CLIENTS adds a `_nc<N>` suffix.
import glob
from IPython.display import Image, display

table_suffix = '' if CUT == 'middle' else f'_{CUT}'
plots_dir_name = 'plots_mnist' if CUT == 'middle' else f'plots_mnist_{CUT}'
if NUM_CLIENTS is not None:
    table_suffix += f'_nc{NUM_CLIENTS}'
    plots_dir_name += f'_nc{NUM_CLIENTS}'

print(open(f"{REPO_DIR}/inference/benchmark_table_mnist{table_suffix}.txt").read())

for png in sorted(glob.glob(f"{REPO_DIR}/inference/{plots_dir_name}/**/*.png", recursive=True)):
    print(png)
    display(Image(filename=png))

## Running a custom experiment

Use the README's Hydra override syntax to run any method/model/dataset combination,
e.g.:

```python
!python main.py algorithm=fsl_sage model=resnet18 dataset=cifar10 \\
  dataset.distribution=noniid_dirichlet dataset.alpha=0.5 cut=shallow \\
  num_clients=20 save=False device=cuda
```

Add `cut=shallow` / `cut=middle` / `cut=deep` to pick the client/server split
point (default is `middle`, this harness's original behavior); see
`src/hydra_config/cut/*.yaml`. Add `num_clients=<N>` to override the client
count (default is 10).

Results land under
`saves/<algorithm>/<model>/<cut>/<dataset>-<distribution>/.../results.json`
(the client count is folded into that final path segment), which (via
the symlink above) is actually on your Drive at
`fsl-sage-colab/saves/...` so it survives runtime resets.